In [1]:
import sys
print(sys.executable)

j:\1_NLP_Data_Engineer\260423_NLP_W1\.fastapienv\Scripts\python.exe


In [2]:
import json
from fastapi import FastAPI
from pydantic import BaseModel
from enum import Enum
from typing import Optional
from typing import List
from decimal import Decimal
from pydantic import BaseModel, ConfigDict, Field, ValidationError
from fastapi import HTTPException

In [ ]:
import json
from fastapi import FastAPI
from pydantic import BaseModel
from enum import Enum
from typing import Optional
from typing import List
from decimal import Decimal
from pydantic import BaseModel, ConfigDict, Field, ValidationError
from fastapi import HTTPException





# FastAPI is about defining how your server exposes and enforces a contract over HTTP.
app = FastAPI()

products = []
current_id = 1

class Tags(Enum):
    FRUIT = 'fruit'
    FRESH = 'fresh'
    SNACK = 'snack'
    SWEET = 'sweet'
    DAIRY = 'dairy'
    VEGAN = 'vegan'
    MISC = 'misc'

class Supplier(BaseModel):
    name: str 
    contact_email: Optional[str] = None

class Product(BaseModel):
    id: int = Field(gt=0)
    name: str = Field(..., max_length=20)
    price: Decimal = Field(max_digits=5, decimal_places=2)
    is_offer: bool = False
    tags: List[Tags] = Field(default_factory=list)
    supplier: Optional[Supplier] = None
# Regarding default_factory, see: https://dev.to/devasservice/python-trick-using-dataclasses-with-fielddefaultfactory-4159

# A field is required if and only if:
    # it has no default value
    # OR explicitly uses Field(...)
class ProductCreate(BaseModel):
    # model_config = ConfigDict(strict=True)    
    name: str = Field(..., max_length=20)
    price: Decimal = Field(max_digits=5, decimal_places=2)
    is_offer: bool = False
    # default for tags = empty list
    tags: List[Tags] = Field(default_factory=list)
    supplier: Optional[Supplier] = None

class ProductUpdate(BaseModel):
    # model_config = ConfigDict(strict=True)    
    name: str = Field(..., max_length=20)
    price: Decimal = Field(max_digits=5, decimal_places=2)
    is_offer: bool = False
    tags: List[Tags] = Field(default_factory=list)
    supplier: Optional[Supplier] = None

# Defining argument data types in Python, see: https://www.geeksforgeeks.org/python/explicitly-define-datatype-in-a-python-function/

@app.post("/products", response_model=Product, status_code=201)
def create_product(payload: ProductCreate):
    print(type(payload.tags[0]))
    
    global current_id
    # Product requires an ID
    new_product = Product(id=current_id, **payload.dict())
    products.append(new_product)

    current_id += 1
    # return the following response to client:
    return new_product


# Function matches {id} with id argument in read_product(id: int)
@app.get("/products/{id}", response_model=Product)
def read_product(id: int):
    for product in products:
        if product.id == id:
            # return the following response to client:
            return product
    raise HTTPException(status_code=404, detail="Product not found")
    
@app.get("/products", response_model=List[Product])
def read_all_products():
    return products



@app.put("/products/{id}", response_model=Product)
def update_product(id: int, payload: ProductCreate):
    for i, product in enumerate(products):
        if product.id == id:
            updated_product = Product(id=id, **payload.dict())
            products[i] = updated_product
            # return the following response to client in JSON (key-value) format:
            return updated_product
    raise HTTPException(status_code=404, detail="Product not found")

@app.delete("/products/{id}", response_model=Product)
def delete_product(id: int):
    for i, product in enumerate(products):
        if product.id == id:
            return products.pop(i)
    raise HTTPException(status_code=404, detail="Product not found")

# This is fine, but: lookup = O(n) (linear search); i.e. not scalable
# Later: dict or database; But don’t change it yet. You’re still learning the mechanics.

# NEXT: Implement PUT /products/{id} using a Python list without breaking your current logic (dict would also be possible but is avoided due to flatter learning curve for you).


# @app.put("/products/{id}", response_model=Product)
# def update_product(id: int, update_product: ProductCreate):
#     i = 0
#     for product in products:
#         if product.id == id:
#             product = Product(id=id, **update_product.dict())
#             products[i] = product
#             # return the following response to client in JSON (key-value) format:
#             return product
#         # increment i in any case until product is returned or exception is raised
#         i += 1
#     raise HTTPException(status_code=404, detail="Product not found")

# @app.delete("/products/{id}", response_model=Product)
# def delete_product(id: int):
#     i=0
#     for product in products:
#         if product.id == id:
#             products.pop(i)
#             # the pop() method by default return the deleted index
#             return products.pop(i)
#         i +=1
#     raise HTTPException(status_code=404, detail="Product not found")




# @app.patch("products/{id}")
# def update_fields(id: int, name | None = None, price | None = None, is_offer | None = None, tag_dict_items | None = None, supplier | None = None):
#     for product in products:
#         if product.id == id:
#             if name is not None:
#                 product.name = name
#             if price is not None:
#                 product.price = price
#             if is_offer is not None:
#                 product.is_offer = is_offer
#             if tag_dict_items is not None:
#                 product.tags = tag_dict_items
#             if name is not None:
#                 product.name = name
#             if supplier is not None:
#                 product.supplier = supplier


In [3]:
print(type(product.tags[0]))

NameError: name 'product' is not defined

In [48]:

with open ("week1.json") as f:
    jsondata = json.load(f)
    print("This is the first item from a dict:")
    print(jsondata[0])
    print("\n")
for data in jsondata:
    print("This is a list:")
    print(data)
    print("\n")

for data in jsondata:    
    for i, value in data.items():
        
        print(f"This is a key for the item: {i}")
        print(f"This is a value for the item {i}: {value}")
        
    #     print(f"value: {value}")
    # supplier = data.get("supplier")
    # print(supplier)
    #tags = data.get("tags")
    # if tags:
    #     for tag in tags:
    #         print(tag)



This is the first item from a dict:
{'id': 1, 'name': 'Apple', 'price': 1.2, 'is_offer': False, 'tags': ['fruit', 'fresh'], 'supplier': {'name': 'Fresh Farms Ltd.', 'contact_email': 'contact@freshfarms.com'}}


This is a list:
{'id': 1, 'name': 'Apple', 'price': 1.2, 'is_offer': False, 'tags': ['fruit', 'fresh'], 'supplier': {'name': 'Fresh Farms Ltd.', 'contact_email': 'contact@freshfarms.com'}}


This is a list:
{'id': 2, 'name': 'Banana', 'price': 0.8, 'is_offer': True, 'tags': ['fruit'], 'supplier': {'name': 'Tropical Goods Inc.', 'contact_email': 'sales@tropicalgoods.com'}}


This is a list:
{'id': 3, 'name': 'Chocolate Bar', 'price': '2.5', 'is_offer': False, 'tags': ['snack', 'sweet'], 'supplier': {'name': 'Sweet Factory', 'contact_email': 'info@sweetfactory.com'}}


This is a list:
{'id': 4, 'name': 'Milk', 'price': 1.5, 'tags': ['dairy'], 'supplier': {'name': 'Dairy Best'}}


This is a list:
{'id': 5, 'name': 'Eggs', 'price': 2.2, 'is_offer': True, 'tags': [], 'supplier': None

In [ ]:
# json.load() loads a JSON file to convert it into a Pyton dict; by contract json.loads reads in a JSON string (see https://note.nkmk.me/en/python-json-load-dump/)

with open ("week1.json") as f:
    jsondata = json.load(f)
    print(f"Anzahl Datensätze: {len(jsondata)}")
    

for dataset in jsondata:
    print(dataset)
    print("\n")

Anzahl Datensätze: 5
{'id': 1, 'name': 'Apple', 'price': 1.2, 'is_offer': False, 'tags': ['fruit', 'fresh'], 'supplier': {'name': 'Fresh Farms Ltd.', 'contact_email': 'contact@freshfarms.com'}}


{'id': 2, 'name': 'Banana', 'price': 0.8, 'is_offer': True, 'tags': ['fruit'], 'supplier': {'name': 'Tropical Goods Inc.', 'contact_email': 'sales@tropicalgoods.com'}}


{'id': 3, 'name': 'Chocolate Bar', 'price': '2.5', 'is_offer': False, 'tags': ['snack', 'sweet'], 'supplier': {'name': 'Sweet Factory', 'contact_email': 'info@sweetfactory.com'}}


{'id': 4, 'name': 'Milk', 'price': 1.5, 'tags': ['dairy'], 'supplier': {'name': 'Dairy Best'}}


{'id': 5, 'name': 'Eggs', 'price': 2.2, 'is_offer': True, 'tags': [], 'supplier': None}




In [ ]:

# Loop through list (incl. dict) loaded from JSON file    
for data in jsondata:
    
    print(f"'id': {data['id']}")
    print(f"'name: {data['name']}")
    print(f"'price' {data['price']}")
    is_offer = data.get('is_offer')
    if is_offer is None:
        print("is_offer: No field or data available")
    else:
        print(f"'is_offer': {is_offer}")
    # print(f"'tags': {data['tags']}")
    tags = data.get('tags')
    if tags is None:
        print("No tags specified!")
    elif not tags:
        print("Tags list is empty")
    else:
        for tag in tags:
            print(f"'tag': {tag}")
    supplier = data.get('supplier')
    if supplier is None:
        print("supplier: No field or data available")
    else:
        supplier_name = supplier.get('name')
        if supplier_name is None:
            print(f"Supplier name not specified!")
        else: 
            print(f"'supplier name': {supplier_name}")
        supplier_email = supplier.get('contact_email')
        if supplier_email is None:
            print(f"Supplier e-mail not specified!")
        else: 
            print(f"'contact_email': {supplier_email}")
    print("")



'id': 1
'name: Apple
'price' 1.2
'is_offer': False
'tag': fruit
'tag': fresh
'supplier name': Fresh Farms Ltd.
'contact_email': contact@freshfarms.com

'id': 2
'name: Banana
'price' 0.8
'is_offer': True
'tag': fruit
'supplier name': Tropical Goods Inc.
'contact_email': sales@tropicalgoods.com

'id': 3
'name: Chocolate Bar
'price' 2.5
'is_offer': False
'tag': snack
'tag': sweet
'supplier name': Sweet Factory
'contact_email': info@sweetfactory.com

'id': 4
'name: Milk
'price' 1.5
is_offer: No field or data available
'tag': dairy
'supplier name': Dairy Best
Supplier e-mail not specified!

'id': 5
'name: Eggs
'price' 2.2
'is_offer': True
Tags list is empty
supplier: No field or data available



In [ ]:
for data in jsondata:
    for key, value in data.items():
        print(key, value)

id 1
name Apple
price 1.2
is_offer False
tags ['fruit', 'fresh']
supplier {'name': 'Fresh Farms Ltd.', 'contact_email': 'contact@freshfarms.com'}
id 2
name Banana
price 0.8
is_offer True
tags ['fruit']
supplier {'name': 'Tropical Goods Inc.', 'contact_email': 'sales@tropicalgoods.com'}
id 3
name Chocolate Bar
price 2.5
is_offer False
tags ['snack', 'sweet']
supplier {'name': 'Sweet Factory', 'contact_email': 'info@sweetfactory.com'}
id 4
name Milk
price 1.5
tags ['dairy']
supplier {'name': 'Dairy Best'}
id 5
name Eggs
price 2.2
is_offer True
tags []
supplier None


# Load the above list/dict/JSON content into a Pydantic object based on the above model


## See: https://pydantic.dev/docs/validation/latest/concepts/models/